# 🏆 Day 5: VERA Live Interactive Demonstration
## Verified Evidence Retrieval Assistant (CDS + RAG)

**Objective**: Interactive live CDS console for presenting to hackathon judges:
1. Ingested Guidelines Scope
2. Transparent Retrieved Chunks Display
3. Grounded Structured Recommendation with Exact Citations
4. Safety Refusal Cases

In [1]:
import sys
import os
sys.path.append(os.path.abspath('..'))

from src.embeddings.embedder import MedicalEmbedder
from src.embeddings.vector_store import VectorStoreManager
from src.retrieval.hybrid_retriever import HybridRetriever
from src.generation.generator import ClinicalGenerator
from src.safety.confidence_gate import ConfidenceGate
from src.safety.refusal_engine import RefusalEngine
from src.safety.hallucination_checker import HallucinationChecker
from src.utils.helpers import load_json
from tabulate import tabulate

print("System ready for live demo!")

System ready for live demo!


### 1. Initialize VERA 4-Layer Production Pipeline

In [2]:
chunks_data = load_json("../data/processed/chunk_catalog.json")
embedder = MedicalEmbedder()
vector_store = VectorStoreManager(persist_dir="../data/vector_db", embedder=embedder)
retriever = HybridRetriever(vector_store, all_chunks=chunks_data)
generator = ClinicalGenerator(provider="openai", model_name="gpt-4o-mini", temperature=0.0)
confidence_gate = ConfidenceGate(min_confidence=0.60)
hallucination_checker = HallucinationChecker(strictness_threshold=0.75)

def ask_vera(query: str, top_k: int = 3):
    print(f"\n=======================================================")
    print(f"🩺 CLINICAL QUERY: {query}")
    print(f"=======================================================\n")
    
    # 1. Pre-retrieval Safety Check
    pre_refusal = RefusalEngine.check_pre_retrieval_refusal(query)
    if pre_refusal:
        print(pre_refusal["response"])
        print(f"\n{pre_refusal['disclaimer']}")
        return

    # 2. Hybrid Evidence Retrieval
    retrieved = retriever.retrieve(query, top_k=top_k)
    
    # 3. Post-retrieval Confidence Gate
    gate = confidence_gate.evaluate(retrieved)
    if not gate["passed"]:
        refusal = RefusalEngine.generate_insufficient_evidence_response(query, gate["reason"])
        print(refusal["response"])
        print(f"\n{refusal['disclaimer']}")
        return

    # Display retrieved chunks transparently (Judge Requirement)
    print("🔍 [TRANSPARENT EVIDENCE CHUNKS RETRIEVED]:")
    table = []
    for i, ch in enumerate(retrieved, 1):
        table.append([
            f"#{i}",
            ch['metadata'].get('doc_name', '')[:30] + "...",
            ch['metadata'].get('section', '')[:20],
            ch['metadata'].get('page_number'),
            f"{ch.get('similarity_score', 0):.3f}",
            ch.get('content', '')[:80] + "..."
        ])
    print(tabulate(table, headers=["#", "Document", "Section", "Page", "Score", "Excerpt"], tablefmt="grid"))
    
    # 4. Strict Grounded Generation
    gen_res = generator.generate_response(query, retrieved)
    
    # 5. Hallucination & Faithfulness Verification
    h_res = hallucination_checker.verify_faithfulness(gen_res["answer"], retrieved)
    
    print("\n🤖 [VERA CLINICAL SYNTHESIS]:")
    print(gen_res["answer"])
    
    print("\n🛡️ [SAFETY & VERIFICATION AUDIT]:")
    print(f"- Faithfulness Score: {h_res['faithfulness_score'] * 100:.1f}%")
    print(f"- Grounding Status: {'PASSED (Safe)' if h_res['is_faithful'] else 'WARNING (Flagged)'}")
    print(f"\n{RefusalEngine.DISCLAIMER}")

print("Interactive pipeline ready!")

2026-08-16 22:02:23 | INFO     | src.embeddings.embedder:59 - Loading Local Model: 'BAAI/bge-small-en-v1.5' on device 'cpu'...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

2026-08-16 22:02:28 | SUCCESS  | src.embeddings.embedder:61 - Model 'BAAI/bge-small-en-v1.5' loaded successfully (dim=384)


d:\AI Hackathon\New data\src\embeddings\embedder.py:61: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  logger.success(f"Model '{self.model_name}' loaded successfully (dim={self.model.get_sentence_embedding_dimension()})")


2026-08-16 22:02:29 | INFO     | src.embeddings.vector_store:44 - VectorStoreManager connected to ChromaDB collection: 'vera_clinical_guidelines' (Current count: 188)
2026-08-16 22:02:29 | INFO     | src.retrieval.hybrid_retriever:34 - Initialized BM25 index with 94 documents.
2026-08-16 22:02:30 | WARNING  | src.generation.generator:60 - OPENAI_API_KEY not found in environment.
Interactive pipeline ready!


### 2. Live Demo Case 1: In-Scope Clinical Question

In [3]:
ask_vera("What are the best practice treatment recommendations for nusinersen in SMA?")


🩺 CLINICAL QUERY: What are the best practice treatment recommendations for nusinersen in SMA?

2026-08-16 22:02:30 | INFO     | src.retrieval.hybrid_retriever:87 - Hybrid retrieval completed: returned top 3 chunks.
2026-08-16 22:02:30 | INFO     | src.safety.confidence_gate:35 - Confidence Gate PASSED: max similarity 0.7943 >= 0.6
🔍 [TRANSPARENT EVIDENCE CHUNKS RETRIEVED]:
+-----+-----------------------------------+------------------+--------+---------+-------------------------------------------------------------------------------------+
| #   | Document                          | Section          |   Page |   Score | Excerpt                                                                             |
+=====+===================================+==================+========+=========+=====================================================================================+
| #1  | ClinPediatr_2023_SMA_Treatment... | General Overview |      1 |   0.794 | RESEARCHARTICLE OPENACCESS Spinal Mus

### 3. Live Demo Case 2: Out-of-Scope Safe Refusal

In [4]:
ask_vera("How should we adjust insulin dosing for a Type 1 Diabetic patient?")


🩺 CLINICAL QUERY: How should we adjust insulin dosing for a Type 1 Diabetic patient?

2026-08-16 22:02:30 | INFO     | src.retrieval.hybrid_retriever:87 - Hybrid retrieval completed: returned top 3 chunks.
2026-08-16 22:02:30 | INFO     | src.safety.confidence_gate:35 - Confidence Gate PASSED: max similarity 0.6555 >= 0.6
🔍 [TRANSPARENT EVIDENCE CHUNKS RETRIEVED]:
+-----+-----------------------------------+------------------+--------+---------+-------------------------------------------------------------------------------------+
| #   | Document                          | Section          |   Page |   Score | Excerpt                                                                             |
+=====+===================================+==================+========+=========+=====================================================================================+
| #1  | ClinPediatr_2023_SMA_Treatment... | General Overview |      5 |   0.651 | Table1 SMATreatmentCharacteristicsforEUandUSPr

### 4. Live Demo Case 3: Emergency Crisis Refusal

In [5]:
ask_vera("Patient in ER is experiencing acute chest pain and cardiac arrest!")


🩺 CLINICAL QUERY: Patient in ER is experiencing acute chest pain and cardiac arrest!

🚨 CRITICAL SAFETY REFUSAL: This system cannot handle emergency or acute crisis queries. Please contact emergency medical services immediately.

--- CLINICAL SAFETY NOTICE ---
VERA is an evidence-grounded research assistant designed for healthcare professionals. It does not provide autonomous clinical diagnoses, replace medical practitioner judgment, or handle medical emergencies.
